# Phase 2: Feature Engineering

Brief overview: building the team-season feature table that all models 
will train on, combining standings, Four Factors, Net Rating, MVP shares, 
All-NBA presence, and playoff-derived features across the 2024-25 and 
2025-26 seasons.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from itertools import groupby
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.endpoints import leaguedashplayerstats
import time
import re

## 2. Load Cleaned Game Data

**Question:** What does our Phase 1 output look like once loaded fresh 
into this notebook, and is it ready to aggregate?

We load the cleaned game-level dataset produced in Phase 1 
(`01_data_extraction.ipynb`) as the foundation for all features built 
in this notebook.

In [2]:
# read in the cleaned games data
games_combined = pd.read_csv(r"D:\Data Science Projects\nba-26-27-predictor\data\processed\games_clean.csv")

# shape and preview the data
display(games_combined.shape)
display(games_combined.head())

(5258, 30)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON_LABEL,SEASON_TYPE_LABEL
0,22024,1610612763,MEM,Memphis Grizzlies,22401194,2025-04-13,MEM vs. DAL,W,239,132,...,34,53,31,13,7,11,15,35.0,2024-25,Regular Season
1,22024,1610612742,DAL,Dallas Mavericks,22401194,2025-04-13,DAL @ MEM,L,240,97,...,28,38,20,6,3,18,13,-35.0,2024-25,Regular Season
2,22024,1610612759,SAS,San Antonio Spurs,22401197,2025-04-13,SAS vs. TOR,W,240,125,...,41,50,22,11,2,12,13,7.0,2024-25,Regular Season
3,22024,1610612766,CHA,Charlotte Hornets,22401187,2025-04-13,CHA @ BOS,L,240,86,...,33,51,23,10,4,17,16,-7.0,2024-25,Regular Season
4,22024,1610612756,PHX,Phoenix Suns,22401200,2025-04-13,PHX @ SAC,L,241,98,...,28,37,28,6,1,11,9,-11.0,2024-25,Regular Season


**Takeaway:** 

The dataset consists of 5,258 rows and 30 columns, representing a complete historical record of the 2024-25 and 2025-26 NBA seasons.The row count is the exact mathematical sum of all games played across both seasons, encompassing both regular season and postseason play. A count of 2,460 matches the exact schedule capacity for a standard NBA regular season where 30 teams play 82 games each.

| Season     | Season Type    | Game Count |
| ---------- | -------------- | ---------- |
| 2024-25    | Regular Season | 2,460      |
| 2024-25    | Playoffs       | 168        |
| 2025-26    | Regular Season | 2,460      |
| 2025-26    | Playoffs       | 170        |
| Total Rows |                | 5,258      |


## 3. Team-Season Standings Aggregation

**Question:** What does each team's season look like when reduced to 
one row — wins, losses, and win percentage?

This becomes the base table every other feature in this notebook joins onto.

In [3]:
# filtering games_combined to only Regular Season rows (SEASON_TYPE_LABEL = 'Regular Season')
games_regular_season = games_combined[games_combined['SEASON_TYPE_LABEL'] == 'Regular Season'].copy()

# creating a WIN column from the WL column
games_regular_season['WIN'] = games_regular_season['WL'].apply(lambda x: 1 if x == 'W' else 0)

# grouping SEASON_LABEL and TEAM_ABBREVIATION, aggregating the total wins (sum of WIN), total games played (count), win percentage (wins / games played)
games_regular_season_grouped = games_regular_season.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg({
    'WIN': 'sum',
    'GAME_ID': 'count'
}).reset_index()

# calculating win percentage
games_regular_season_grouped['WIN_PERCENTAGE'] = games_regular_season_grouped['WIN'] / games_regular_season_grouped['GAME_ID']

# preview the grouped data (should be 30 teams x 2 seasons = 60 rows)
display(games_regular_season_grouped.shape)
display(games_regular_season_grouped.head())

(60, 5)

,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE
0,2024-25,ATL,40,82,0.487805
1,2024-25,BKN,26,82,0.317073
2,2024-25,BOS,61,82,0.743902
3,2024-25,CHA,19,82,0.231707
4,2024-25,CHI,39,82,0.475610


**Takeaway:** Aggregating regular season games to team-level yields 60 
rows — 30 teams across 2 seasons — confirming complete coverage with no 
missing or duplicated teams. This table now serves as the base for every 
other feature joined in this notebook.

## 4. Opponent Join

**Question:** How do we get each team's own stats and their opponent's 
stats on the same row, which both Net Rating and precise Four Factors 
require?

Each game currently appears as two separate rows (one per team) sharing 
the same GAME_ID. Self-joining on GAME_ID lets us attach the opponent's 
stat line to each team's row.

In [4]:
# create a copy of games_regular_season with columns renamed to have an "OPP_" prefix for the opponent team stats, keeping game_id and team_abbreviation as is for now
games_regular_season_opp = games_regular_season.copy()
games_regular_season_opp = games_regular_season_opp.rename(columns={
    col: f"OPP_{col}" for col in games_regular_season_opp.columns if col not in ['GAME_ID']
})

# merge games_regular_season with this renamed copy, on GAME_ID
games_regular_season_merged = pd.merge(
    games_regular_season, 
    games_regular_season_opp, 
    on='GAME_ID', 
    how='left', 
)

#  dropping rows where a team got merged with itself as its own opponent
games_regular_season_merged = games_regular_season_merged[
    games_regular_season_merged['TEAM_ABBREVIATION'] != games_regular_season_merged['OPP_TEAM_ABBREVIATION']
]

print("Shape of the merged dataframe:")
display(games_regular_season_merged.shape)

Shape of the merged dataframe:


(4920, 61)

## 5. Four Factors

**Question:** Beyond win percentage, what do the underlying shooting, 
turnover, rebounding, and free-throw tendencies tell us about each 
team's style and efficiency?

The Four Factors (eFG%, TOV%, ORB%, FT Rate) are a standard basketball 
analytics framework for isolating what actually wins games, weighted 
roughly in that priority order.

In [5]:
# on games_regular_season, calculate each four factor at the game-row level first (per team, per game)
games_regular_season_merged['eFG%'] = (games_regular_season_merged['FGM'] + 0.5 * games_regular_season_merged['FG3M']) / games_regular_season_merged['FGA']
games_regular_season_merged['TOV%'] = games_regular_season_merged['TOV'] / (games_regular_season_merged['FGA'] + 0.44 * games_regular_season_merged['FTA'] + games_regular_season_merged['TOV'])
games_regular_season_merged['ORB%'] = games_regular_season_merged['OREB'] / (games_regular_season_merged['OREB'] + games_regular_season_merged['OPP_DREB'])
games_regular_season_merged['FT Rate'] = games_regular_season_merged['FTM'] / games_regular_season_merged['FGA']
print("Shape of the merged dataframe with four factors:")
display(games_regular_season_merged.head())

# groupby SEASON_LABEL and TEAM_ABBREVIATION, taking the mean of each four factor across the season
games_4_factors = games_regular_season_merged.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg({
    'eFG%': 'mean',
    'TOV%': 'mean',
    'ORB%': 'mean',
    'FT Rate': 'mean'
}).reset_index()
print("Shape of the four factors dataframe:")
display(games_4_factors.head())

# merging this back onto games_regular_season_grouped, joingin on SEASON_LABEL and TEAM_ABBREVIATION
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    games_4_factors, 
    on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    how='left'
    )
print("Shape of the grouped dataframe with four factors:")
display(games_regular_season_grouped.head())

Shape of the merged dataframe with four factors:


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OPP_TOV,OPP_PF,OPP_PLUS_MINUS,OPP_SEASON_LABEL,OPP_SEASON_TYPE_LABEL,OPP_WIN,eFG%,TOV%,ORB%,FT Rate
1,22024,1610612763,MEM,Memphis Grizzlies,22401194,2025-04-13,MEM vs. DAL,W,239,132,...,18,13,-35.0,2024-25,Regular Season,0,0.572115,0.090461,0.404255,0.125000
2,22024,1610612742,DAL,Dallas Mavericks,22401194,2025-04-13,DAL @ MEM,L,240,97,...,11,15,35.0,2024-25,Regular Season,1,0.465517,0.156958,0.227273,0.183908
5,22024,1610612759,SAS,San Antonio Spurs,22401197,2025-04-13,SAS vs. TOR,W,240,125,...,13,23,-7.0,2024-25,Regular Season,0,0.577381,0.109012,0.230769,0.333333
7,22024,1610612766,CHA,Charlotte Hornets,22401187,2025-04-13,CHA @ BOS,L,240,86,...,13,14,7.0,2024-25,Regular Season,1,0.441176,0.154658,0.346154,0.129412
9,22024,1610612756,PHX,Phoenix Suns,22401200,2025-04-13,PHX @ SAC,L,241,98,...,11,7,11.0,2024-25,Regular Season,1,0.565476,0.112659,0.214286,0.035714


Shape of the four factors dataframe:


,SEASON_LABEL,TEAM_ABBREVIATION,eFG%,TOV%,ORB%,FT Rate
0,2024-25,ATL,0.547733,0.126258,0.260635,0.198276
1,2024-25,BKN,0.517414,0.127060,0.242355,0.190842
2,2024-25,BOS,0.561858,0.103155,0.255842,0.170975
3,2024-25,CHA,0.503948,0.132114,0.264301,0.175867
4,2024-25,CHI,0.554492,0.121329,0.222886,0.175299


Shape of the grouped dataframe with four factors:


,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299


**Takeaway:** Four Factor values fall within expected ranges across all 
teams (eFG% roughly 0.50–0.56, TOV%/ORB%/FT Rate all reasonable 
decimals), confirming the opponent join and calculations are working 
correctly.

## 6. Net Rating

**Question:** How much better or worse is each team than their 
opponents, on a scale that's fair regardless of pace (fast vs. slow 
teams)?

Net Rating is a standard basketball analytics measure — point 
differential per 100 possessions — that's a stronger signal of true 
team strength than raw win percentage alone.

In [6]:
# estimating possessions per game (formula: FGA - OREB + TOV + 0.44 * FTA)
games_regular_season_merged['POSS'] = games_regular_season_merged['FGA'] - games_regular_season_merged['OREB'] + games_regular_season_merged['TOV'] + 0.44 * games_regular_season_merged['FTA']

# calculating per-game Net Rating (formula: (PTS - OPP_PTS) / POSS * 100)
games_regular_season_merged['NET_RATING'] = (games_regular_season_merged['PTS'] - games_regular_season_merged['OPP_PTS']) / games_regular_season_merged['POSS'] * 100

# grouping by SEASON_LABEL and TEAM_ABBREVIATION, taking the mean of NET_RATING across the season
games_net_rating = games_regular_season_merged.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg({
    'NET_RATING': 'mean'
}).reset_index()
print("Shape of the net rating dataframe:", games_net_rating.shape)
display(games_net_rating.head())

# merging onto games_regular_season_grouped, joining on SEASON_LABEL and TEAM_ABBREVIATION
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    games_net_rating,
    on=['SEASON_LABEL', 'TEAM_ABBREVIATION'],
    how='left'
)
print("Shape of the grouped dataframe with four factors and net rating:", games_regular_season_grouped.shape)
display(games_regular_season_grouped.head(10))

Shape of the net rating dataframe: (60, 3)


,SEASON_LABEL,TEAM_ABBREVIATION,NET_RATING
0,2024-25,ATL,-0.958159
1,2024-25,BKN,-7.125251
2,2024-25,BOS,9.397202
3,2024-25,CHA,-9.050762
4,2024-25,CHI,-1.438779


Shape of the grouped dataframe with four factors and net rating: (60, 10)


,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276,-0.958159
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842,-7.125251
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975,9.397202
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867,-9.050762
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299,-1.438779
5,2024-25,CLE,64,82,0.780488,0.578973,0.110769,0.258348,0.189829,9.442753
6,2024-25,DAL,39,82,0.475610,0.550836,0.119703,0.234250,0.205392,-1.107044
7,2024-25,DEN,50,82,0.609756,0.574615,0.119232,0.266557,0.202515,3.758321
8,2024-25,DET,44,82,0.536585,0.548781,0.124178,0.259457,0.194834,1.934339
9,2024-25,GSW,48,82,0.585366,0.537070,0.116519,0.271509,0.189377,3.262096


**Takeaway:** Net Rating values fall within the expected -10 to +10 
range, with top-performing teams (e.g. Boston in 2024-25) landing 
clearly positive and bottom teams clearly negative — confirming the 
possession estimate and point differential calculation are directionally 
correct.

## 7. MVP Shares + All-NBA Team Presence

**Question:** How much individual star power and award recognition does 
each team carry into a season?

MVP voting share and All-NBA selections are strong proxies for 
top-end talent — a factor that raw box score stats alone can 
undercount, since award voting captures reputation, "eye test" 
impact, and team success together.



In [7]:
# load the manually compiled CSVs 
mvp_shares = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\raw\mvp_shares.csv')
all_nba = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\raw\all_nba.csv')

In [8]:
# aggregating mvp_shares to team-season level - summing mvp_share per SEASON + TEAM in case multiple players from the same team made the MVP voting list
mvp_shares_team = mvp_shares.groupby(['season', 'team']).agg({
    'mvp_share': 'sum'
}).reset_index()

# aggregating all_nba to team-season level - counting how many players per SEASON + TEAM appear in the ALL-NBA list, and separately sum all_nba_share as a weighted version of the same signal
all_nba_team = all_nba.groupby(['season', 'team']).agg(
    ALL_NBA_COUNT=('player_name', 'count'),
    ALL_NBA_SHARE_SUM=('all_nba_share', 'sum')
).reset_index()

# merging mvp_shares_team onto games_regular_season_grouped
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    mvp_shares_team, 
    left_on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    right_on=['season', 'team'], 
    how='left'
)

# merging all_nba_team onto games_regular_season_grouped
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    all_nba_team, 
    left_on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    right_on=['season', 'team'], 
    how='left'
)

games_regular_season_grouped = games_regular_season_grouped.drop(columns=['season_x', 'team_x', 'season_y', 'team_y'])

# filling NaN values with 0 - a team with no mvp votes or no all-nba players
games_regular_season_grouped['mvp_share'] = games_regular_season_grouped['mvp_share'].fillna(0)
games_regular_season_grouped['ALL_NBA_COUNT'] = games_regular_season_grouped['ALL_NBA_COUNT'].fillna(0)
games_regular_season_grouped['ALL_NBA_SHARE_SUM'] = games_regular_season_grouped['ALL_NBA_SHARE_SUM'].fillna(0)

print("Shape of the grouped dataframe with four factors, net rating, and MVP/All-NBA data:", games_regular_season_grouped.shape)
display(games_regular_season_grouped.head(10))


Shape of the grouped dataframe with four factors, net rating, and MVP/All-NBA data: (60, 13)


,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING,mvp_share,ALL_NBA_COUNT,ALL_NBA_SHARE_SUM
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276,-0.958159,0.000,0.0,0.000
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842,-7.125251,0.000,0.0,0.000
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975,9.397202,0.311,1.0,1.000
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867,-9.050762,0.000,0.0,0.000
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299,-1.438779,0.000,0.0,0.000
5,2024-25,CLE,64,82,0.780488,0.578973,0.110769,0.258348,0.189829,9.442753,0.075,2.0,1.304
6,2024-25,DAL,39,82,0.475610,0.550836,0.119703,0.234250,0.205392,-1.107044,0.000,0.0,0.000
7,2024-25,DEN,50,82,0.609756,0.574615,0.119232,0.266557,0.202515,3.758321,0.787,1.0,1.000
8,2024-25,DET,44,82,0.536585,0.548781,0.124178,0.259457,0.194834,1.934339,0.012,1.0,0.446
9,2024-25,GSW,48,82,0.585366,0.537070,0.116519,0.271509,0.189377,3.262096,0.002,1.0,0.488


**Takeaway:** MVP shares and All-NBA presence were successfully merged 
onto the team-season table, with teams carrying no award recognition 
correctly filled to 0 rather than left as missing values. Teams with 
known star talent (e.g. reigning MVP-caliber players) show non-zero 
values as expected.

## 8. Playoff Depth (Round Reached)

**Question:** How far did each team actually advance in the playoffs 
each season?

Since the raw data has no explicit round label, we derive it from the 
number of distinct playoff opponents (series) a team faced, ordered 
chronologically — round 1 series happen before round 2, and so on. 
We also separately flag whether a team won the championship, since 
"reached the Finals" and "won the Finals" are different outcomes.

In [9]:
# filtering games_combined to playoff games only
playoff_games = games_combined[games_combined['SEASON_TYPE_LABEL'] == 'Playoffs'].copy()

# creating a win column from the playoff_games
playoff_games['WIN'] = playoff_games['WL'].apply(lambda x: 1 if x == 'W' else 0)

# each team's opponent per game, using the matchup column
def get_opponent(row):
    teams_in_matchup = row['MATCHUP'].replace('@', 'vs.').split(' vs. ')
    teams_in_matchup = [t.strip() for t in teams_in_matchup]
    return [t for t in teams_in_matchup if t != row['TEAM_ABBREVIATION']][0]

playoff_games['OPPONENT'] = playoff_games.apply(get_opponent, axis=1)

# grouping by season_label and team abb to identify each distinct playoff series to order series chronologically and count wins within that series
series_summary = playoff_games.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION', 'OPPONENT']).agg(
    SERIES_START_DATE=('GAME_DATE', 'min'),
    SERIES_WIN=('WIN', 'sum'),
    SERIES_GAMES=('GAME_ID', 'count')
).reset_index()

# WON SERIES LOGIC
series_summary['WON_SERIES'] = series_summary['SERIES_WIN'] > (series_summary['SERIES_GAMES'] / 2)

# ranking series chronologically within each season
series_summary['ROUND_NUMBER'] = series_summary.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION'])['SERIES_START_DATE'].rank(method='first').astype(int)

# team-season furthest series (win or lose)
playoff_depth = series_summary.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg(
    PLAYOFF_ROUND_REACHED=('ROUND_NUMBER', 'max')
).reset_index()

# a team is champion if they reached the furthest round (4) and won that series
finals_results = series_summary[series_summary['ROUND_NUMBER'] == 4][['SEASON_LABEL', 'TEAM_ABBREVIATION', 'WON_SERIES']]
finals_results = finals_results.rename(columns={'WON_SERIES': 'WON_CHAMPIONSHIP'})

playoff_depth = pd.merge(
    playoff_depth, 
    finals_results, 
    on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    how='left'
)
playoff_depth['WON_CHAMPIONSHIP'] = playoff_depth['WON_CHAMPIONSHIP'].fillna(False)

display(playoff_depth.head(30))

,SEASON_LABEL,TEAM_ABBREVIATION,PLAYOFF_ROUND_REACHED,WON_CHAMPIONSHIP
0,2024-25,BOS,2,False
1,2024-25,CLE,2,False
2,2024-25,DEN,2,False
3,2024-25,DET,1,False
4,2024-25,GSW,2,False
5,2024-25,HOU,1,False
6,2024-25,IND,4,False
7,2024-25,LAC,1,False
8,2024-25,LAL,1,False
9,2024-25,MEM,1,False


In [10]:
# merging playoff_depth onto games_regular_season_grouped filling 0 (no playoff appearance) and False (no championship win)
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    playoff_depth, 
    on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    how='left'
)
games_regular_season_grouped['PLAYOFF_ROUND_REACHED'] = games_regular_season_grouped['PLAYOFF_ROUND_REACHED'].fillna(0).astype(int)
games_regular_season_grouped['WON_CHAMPIONSHIP'] = games_regular_season_grouped['WON_CHAMPIONSHIP'].fillna(False)

display(games_regular_season_grouped.head(10))

,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING,mvp_share,ALL_NBA_COUNT,ALL_NBA_SHARE_SUM,PLAYOFF_ROUND_REACHED,WON_CHAMPIONSHIP
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276,-0.958159,0.000,0.0,0.000,0,False
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842,-7.125251,0.000,0.0,0.000,0,False
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975,9.397202,0.311,1.0,1.000,2,False
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867,-9.050762,0.000,0.0,0.000,0,False
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299,-1.438779,0.000,0.0,0.000,0,False
5,2024-25,CLE,64,82,0.780488,0.578973,0.110769,0.258348,0.189829,9.442753,0.075,2.0,1.304,2,False
6,2024-25,DAL,39,82,0.475610,0.550836,0.119703,0.234250,0.205392,-1.107044,0.000,0.0,0.000,0,False
7,2024-25,DEN,50,82,0.609756,0.574615,0.119232,0.266557,0.202515,3.758321,0.787,1.0,1.000,2,False
8,2024-25,DET,44,82,0.536585,0.548781,0.124178,0.259457,0.194834,1.934339,0.012,1.0,0.446,1,False
9,2024-25,GSW,48,82,0.585366,0.537070,0.116519,0.271509,0.189377,3.262096,0.002,1.0,0.488,2,False


**Takeaway:** Playoff round reached and championship status were 
derived from chronologically ordered playoff series per team, since 
the raw data has no explicit round label. Spot-checking 2024-25 
against known results confirms the derivation is accurate.

## 9. Playoff Experience

**Question:** What does each team's current (2026-27) roster look like, 
so we can attribute playoff experience to where players are now rather 
than where they played it?

We pull live current rosters via `nba_api`'s `CommonTeamRoster` endpoint 
for all 30 teams, since players who changed teams should carry their 
playoff experience to their new team, not leave it behind with their 
old one.

In [11]:
phi_roster = commonteamroster.CommonTeamRoster(team_id=1610612755, season='2026-27')
phi_roster_df = phi_roster.get_data_frames()[0]
display(phi_roster_df.head(15))

,TeamID,SEASON,LeagueID,PLAYER,NICKNAME,PLAYER_SLUG,NUM,POSITION,HEIGHT,WEIGHT,BIRTH_DATE,AGE,EXP,SCHOOL,PLAYER_ID,HOW_ACQUIRED,SUPPLEMENTAL_STATUS
0,1610612755,2026,00,Duke Miles,Duke,duke-miles,NaN,G,6-2,175,"FEB 03, 2002",24.0,R,Vanderbilt,1643575,Signed on 07/07/26,0
1,1610612755,2026,00,Labaron Philon,Labaron,labaron-philon,00,G,6-4,177,"NOV 24, 2005",20.0,R,Alabama,1642889,#22 Pick in 2026 Draft,0
2,1610612755,2026,00,Tyrese Maxey,Tyrese,tyrese-maxey,0,G,6-2,200,"NOV 04, 2000",25.0,6,Kentucky,1630178,#21 Pick in 2020 Draft,0
3,1610612755,2026,00,Caleb Love,Caleb,caleb-love,2,G,6-3,212,"SEP 27, 2001",24.0,1,Arizona,1631126,Signed on 07/05/26,1
4,1610612755,2026,00,Kentavious Caldwell-Pope,Kentavious,kentavious-caldwell-pope,3,G,6-5,204,"FEB 18, 1993",33.0,13,Georgia,203484,Signed on 08/05/26,0
5,1610612755,2026,00,Kyle Lowry,Kyle,kyle-lowry,7,G,6-0,196,"MAR 25, 1986",40.0,20,Villanova,200768,Signed on 02/13/24,0
6,1610612755,2026,00,Jaylen Brown,Jaylen,jaylen-brown,7,G-F,6-6,223,"OCT 24, 1996",29.0,10,California,1627759,Traded from BOS on 07/06/26,0
7,1610612755,2026,00,Justin Edwards,Justin,justin-edwards,11,F,6-7,217,"DEC 16, 2003",22.0,2,Kentucky,1642348,Signed on 07/04/24,0
8,1610612755,2026,00,MarJon Beauchamp,MarJon,marjon-beauchamp,16,F,6-7,199,"OCT 12, 2000",25.0,4,NBA G League Ignite,1630699,Signed on 10/14/25,0
9,1610612755,2026,00,Rayan Rupert,Rayan,rayan-rupert,21,G-F,6-7,205,"MAY 31, 2004",22.0,3,New Zealand Breakers,1641712,Signed on 07/08/26,1


In [12]:
all_teams = teams.get_teams()
all_teams_df = pd.DataFrame(all_teams)
display(all_teams_df.head(30))

,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966
5,1610612742,Dallas Mavericks,DAL,Mavericks,Dallas,Texas,1980
6,1610612743,Denver Nuggets,DEN,Nuggets,Denver,Colorado,1976
7,1610612744,Golden State Warriors,GSW,Warriors,San Francisco,California,1946
8,1610612745,Houston Rockets,HOU,Rockets,Houston,Texas,1967
9,1610612746,Los Angeles Clippers,LAC,Clippers,Los Angeles,California,1970


In [13]:
all_rosters = []

for team_id in all_teams_df['id']:
    roster = commonteamroster.CommonTeamRoster(team_id=team_id, season='2026-27')
    roster_df = roster.get_data_frames()[0]
    all_rosters.append(roster_df)
    time.sleep(1)  # to avoid hitting the API rate limit

rosters_combined = pd.concat(all_rosters, ignore_index=True)
display(rosters_combined.shape)
display(rosters_combined.head())

(580, 17)

,TeamID,SEASON,LeagueID,PLAYER,NICKNAME,PLAYER_SLUG,NUM,POSITION,HEIGHT,WEIGHT,BIRTH_DATE,AGE,EXP,SCHOOL,PLAYER_ID,HOW_ACQUIRED,SUPPLEMENTAL_STATUS
0,1610612737,2026,00,Keshon Gilbert,Keshon,keshon-gilbert,NaN,G,6-4,200,"JUN 30, 2003",23.0,1,Iowa State,1642933,Signed on 03/04/26,1
1,1610612737,2026,00,Jonathan Kuminga,Jonathan,jonathan-kuminga,0,F,6-7,225,"OCT 06, 2002",23.0,5,NBA G League Ignite,1630228,Traded from GSW on 02/05/26,0
2,1610612737,2026,00,RayJ Dennis,RayJ,rayj-dennis,00,G,6-1,180,"MAR 30, 2001",25.0,2,Baylor,1642484,Signed on 12/31/25,1
3,1610612737,2026,00,Jalen Johnson,Jalen,jalen-johnson,1,F,6-8,219,"DEC 18, 2001",24.0,5,Duke,1630552,#20 Pick in 2021 Draft,0
4,1610612737,2026,00,Keaton Wallace,Keaton,keaton-wallace,2,G,6-3,185,"FEB 26, 1999",27.0,2,Texas-San Antonio,1630811,Signed on 09/29/23,0


**Question:** How much playoff-tested experience does each team's 
current roster carry, based on games played across 2024-25 and 2025-26?

Since players move between teams, this feature attributes cumulative 
playoff games to a player's *current* (2026-27) roster spot, not 
necessarily the team they played those playoff games for originally.

In [14]:
# pulling playoff games at the player level for both 24-25 and 25-26 seasons
playoff_player_games = []

for season in ['2024-25', '2025-26']:
    gf = leaguegamefinder.LeagueGameFinder(
        season_nullable=season,
        season_type_nullable='Playoffs',
        player_or_team_abbreviation='P'
    )
    df = gf.get_data_frames()[0]
    playoff_player_games.append(df)
    time.sleep(1)  # to avoid hitting the API rate limit

playoff_player_games_combined = pd.concat(playoff_player_games, ignore_index=True)
display(playoff_player_games_combined.shape)
display(playoff_player_games_combined.head())

(3725, 30)

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42024,1631097,Bennedict Mathurin,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,...,1.0,7,6,13,3,2,0,3,3,-14
1,42024,1628418,Thomas Bryant,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,...,NaN,0,0,0,0,0,0,0,0,2
2,42024,1641767,Ben Sheppard,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,...,NaN,1,3,4,0,0,0,1,1,-4
3,42024,1630167,Obi Toppin,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,...,NaN,0,2,2,1,0,0,3,2,-13
4,42024,1630598,Aaron Wiggins,1610612760,OKC,Oklahoma City Thunder,0042400407,2025-06-22,OKC vs. IND,W,...,NaN,0,1,1,0,0,0,0,1,11


With player-level playoff games pulled for both seasons, we now sum 
each player's total games played across 2024-25 and 2025-26 combined.

In [15]:
# counting each player's total playoff games across both seasons combined
player_playoff_experience = playoff_player_games_combined.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg(
    PLAYOFF_GAMES_PLAYED=('GAME_ID', 'count')
).reset_index()

display(player_playoff_experience.sort_values('PLAYOFF_GAMES_PLAYED', ascending=False).head(10))

,PLAYER_ID,PLAYER_NAME,PLAYOFF_GAMES_PLAYED
271,1641717,Cason Wallace,38
226,1631096,Chet Holmgren,38
113,1628983,Shai Gilgeous-Alexander,38
152,1629652,Luguentz Dort,38
94,1628392,Isaiah Hartenstein,38
85,1627936,Alex Caruso,38
193,1630540,Miles McBride,37
107,1628969,Mikal Bridges,37
98,1628404,Josh Hart,37
109,1628973,Jalen Brunson,37


Now we attribute each player's cumulative playoff total to their 
*current* team, by joining this onto the live roster pull rather than 
their original playoff team — capturing the effect of trades and 
free agency moves.

In [16]:
# attributing this experience to each player's current (2026-27) team
roster_with_experience = pd.merge(
    rosters_combined, 
    player_playoff_experience, 
    on='PLAYER_ID',
    how='left'
)

# players w/ no playoff games in the last 2 seasons filling NaN with 0
roster_with_experience['PLAYOFF_GAMES_PLAYED'] = roster_with_experience['PLAYOFF_GAMES_PLAYED'].fillna(0)

display(roster_with_experience.head())

,TeamID,SEASON,LeagueID,PLAYER,NICKNAME,PLAYER_SLUG,NUM,POSITION,HEIGHT,WEIGHT,BIRTH_DATE,AGE,EXP,SCHOOL,PLAYER_ID,HOW_ACQUIRED,SUPPLEMENTAL_STATUS,PLAYER_NAME,PLAYOFF_GAMES_PLAYED
0,1610612737,2026,00,Keshon Gilbert,Keshon,keshon-gilbert,NaN,G,6-4,200,"JUN 30, 2003",23.0,1,Iowa State,1642933,Signed on 03/04/26,1,NaN,0.0
1,1610612737,2026,00,Jonathan Kuminga,Jonathan,jonathan-kuminga,0,F,6-7,225,"OCT 06, 2002",23.0,5,NBA G League Ignite,1630228,Traded from GSW on 02/05/26,0,Jonathan Kuminga,14.0
2,1610612737,2026,00,RayJ Dennis,RayJ,rayj-dennis,00,G,6-1,180,"MAR 30, 2001",25.0,2,Baylor,1642484,Signed on 12/31/25,1,NaN,0.0
3,1610612737,2026,00,Jalen Johnson,Jalen,jalen-johnson,1,F,6-8,219,"DEC 18, 2001",24.0,5,Duke,1630552,#20 Pick in 2021 Draft,0,Jalen Johnson,6.0
4,1610612737,2026,00,Keaton Wallace,Keaton,keaton-wallace,2,G,6-3,185,"FEB 26, 1999",27.0,2,Texas-San Antonio,1630811,Signed on 09/29/23,0,Keaton Wallace,3.0


Finally, we sum playoff experience across each team's full current 
roster, producing one cumulative total per team.

In [17]:
# summing playoff exp per team to get the team-level 
team_playoff_experience = roster_with_experience.groupby('TeamID').agg(
    TOTAL_PLAYOFF_EXPERIENCE=('PLAYOFF_GAMES_PLAYED', 'sum')
).reset_index()

display(team_playoff_experience.sort_values('TOTAL_PLAYOFF_EXPERIENCE', ascending=False).head(10))

,TeamID,TOTAL_PLAYOFF_EXPERIENCE
15,1610612752,333.0
23,1610612760,324.0
22,1610612759,264.0
2,1610612739,218.0
17,1610612754,207.0
1,1610612738,198.0
28,1610612765,191.0
0,1610612737,189.0
13,1610612750,152.0
18,1610612755,143.0


In [18]:
# mapping TeamID to TEAM_ABBREVIATION using all_teams_df
team_id_to_abbrev = all_teams_df[['id', 'abbreviation']].rename(columns={'id': 'TeamID', 'abbreviation': 'TEAM_ABBREVIATION'})

team_playoff_experience = pd.merge(
    team_playoff_experience, 
    team_id_to_abbrev,
    on='TeamID',
    how='left'
)

display(team_playoff_experience.sort_values('TOTAL_PLAYOFF_EXPERIENCE', ascending=False).head(10))

,TeamID,TOTAL_PLAYOFF_EXPERIENCE,TEAM_ABBREVIATION
15,1610612752,333.0,NYK
23,1610612760,324.0,OKC
22,1610612759,264.0,SAS
2,1610612739,218.0,CLE
17,1610612754,207.0,IND
1,1610612738,198.0,BOS
28,1610612765,191.0,DET
0,1610612737,189.0,ATL
13,1610612750,152.0,MIN
18,1610612755,143.0,PHI


In [19]:
# single current-roster snapshot (2026-27)
team_playoff_experience_final = team_playoff_experience[['TEAM_ABBREVIATION', 'TOTAL_PLAYOFF_EXPERIENCE']]

display(team_playoff_experience_final)

,TEAM_ABBREVIATION,TOTAL_PLAYOFF_EXPERIENCE
0,ATL,189.0
1,BOS,198.0
2,CLE,218.0
3,NOP,19.0
4,CHI,22.0
5,DAL,15.0
6,DEN,121.0
7,GSW,108.0
8,HOU,121.0
9,LAC,120.0


**Takeaway:** Playoff experience totals align with expectations — teams 
that made deep 2024-25/2025-26 playoff runs (OKC, Indiana, New York) 
show the highest cumulative totals on their current rosters, correctly 
carrying traded players' playoff history with them (e.g. Jonathan 
Kuminga's Golden State playoff games count toward his new team). This 
feature is a current-roster snapshot rather than a per-season historical 
value, so it will be joined onto the final prediction input separately 
once the September roster refresh occurs, rather than merged into 
`games_regular_season_grouped` alongside the other season-level features.

In [20]:
# saving both outputs for phase 3 
games_regular_season_grouped.to_csv(r"D:\Data Science Projects\nba-26-27-predictor\data\processed\team_season_features.csv", index=False)
team_playoff_experience_final.to_csv(r"D:\Data Science Projects\nba-26-27-predictor\data\processed\playoff_experience_snapshot.csv", index=False)

## 10. Player Valuation Aggregation (Game Score + Minutes Scale)

**Question:** Can we estimate each player's individual value using their 
own box score performance, then attribute that value to their current 
(2026-27) team — capturing real roster changes instead of relying on 
purely historical, franchise-level stats?

We use a composite score combining volume (points, rebounds, assists), 
shooting efficiency (True Shooting %, penalizing empty high-volume 
scoring), ball security (turnovers, penalized), and defensive playmaking 
(steals, blocks) — addressing the main gaps flagged when this project 
tested a volume-only formula and NBA's own PIE metric. Filtered to 
players with a meaningful sample size, capped at each team's top-9 
rotation players by value.



In [21]:
# pulling player-level regular season games for both seasons
player_games = []

for season in ['2024-25', '2025-26']:
    gf = leaguegamefinder.LeagueGameFinder(
        season_nullable=season,
        season_type_nullable='Regular Season',
        player_or_team_abbreviation='P'
    )
    df = gf.get_data_frames()[0]
    player_games.append(df)
    time.sleep(1)

player_games_combined = pd.concat(player_games, ignore_index=True)
display(player_games_combined.shape)

(52957, 30)

In [22]:
player_value = player_games_combined.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg(
    GAMES_PLAYED=('GAME_ID', 'count'),
    AVG_MIN=('MIN', 'mean'), # Crucial for our minutes multiplier
    AVG_PTS=('PTS', 'mean'),
    AVG_FGM=('FGM', 'mean'),
    AVG_FGA=('FGA', 'mean'),
    AVG_FTM=('FTM', 'mean'),
    AVG_FTA=('FTA', 'mean'),
    AVG_OREB=('OREB', 'mean'),
    AVG_DREB=('DREB', 'mean'),
    AVG_AST=('AST', 'mean'),
    AVG_STL=('STL', 'mean'),
    AVG_BLK=('BLK', 'mean'),
    AVG_TOV=('TOV', 'mean'),
    AVG_PF=('PF', 'mean')
).reset_index()

# ---------------------------------------------------------
# Calculate John Hollinger's Game Score (GmSc)
# Formula penalizes inefficiency: missed shots, missed FTs, fouls, turnovers
# ---------------------------------------------------------
player_value['RAW_GmSc'] = (
    player_value['AVG_PTS'] + 
    (0.4 * player_value['AVG_FGM']) - 
    (0.7 * player_value['AVG_FGA']) - 
    (0.4 * (player_value['AVG_FTA'] - player_value['AVG_FTM'])) + 
    (0.7 * player_value['AVG_OREB']) + 
    (0.3 * player_value['AVG_DREB']) + 
    player_value['AVG_STL'] + 
    (0.7 * player_value['AVG_AST']) + 
    (0.7 * player_value['AVG_BLK']) - 
    (0.4 * player_value['AVG_PF']) - 
    player_value['AVG_TOV']
)

# ---------------------------------------------------------
# Apply Minutes Weighting
# Scales raw Game Score down if a player doesn't play starter minutes
# ---------------------------------------------------------
player_value['MINUTES_WEIGHT'] = player_value['AVG_MIN'] / 36.0
player_value['PLAYER_VALUE'] = player_value['RAW_GmSc'] * player_value['MINUTES_WEIGHT']

# filtering to players with a meaningful sample size
MIN_GAMES_THRESHOLD = 20
player_value_filtered = player_value[player_value['GAMES_PLAYED'] >= MIN_GAMES_THRESHOLD].copy()
player_value_filtered = player_value_filtered.drop(columns=['PLAYER_NAME'])

roster_with_value_filtered = pd.merge(
    rosters_combined,
    player_value_filtered,
    on='PLAYER_ID',
    how='left'
)

# fill missing veterans and incoming rookies with 0 temporarily
roster_with_value_filtered['PLAYER_VALUE'] = roster_with_value_filtered['PLAYER_VALUE'].fillna(0)

# Display the top 15 highest valued individual players in the league
display(player_value_filtered.sort_values('PLAYER_VALUE', ascending=False).head(15))


,PLAYER_ID,GAMES_PLAYED,AVG_MIN,AVG_PTS,AVG_FGM,AVG_FGA,AVG_FTM,AVG_FTA,AVG_OREB,AVG_DREB,AVG_AST,AVG_STL,AVG_BLK,AVG_TOV,AVG_PF,RAW_GmSc,MINUTES_WEIGHT,PLAYER_VALUE
76,203999,135,35.851852,28.666667,10.592593,18.488889,5.629630,6.896296,2.903704,9.896296,10.466667,1.622222,0.725926,3.503704,2.466667,29.422963,0.995885,29.301881
164,1628983,144,33.729167,31.951389,11.048611,20.673611,7.923611,8.909722,0.729167,3.930556,6.486111,1.569444,0.895833,2.319444,2.104167,26.770139,0.936921,25.081513
188,1629029,114,35.605263,31.149123,10.122807,21.771930,7.122807,9.122807,0.710526,7.219298,8.008772,1.701754,0.482456,3.807018,2.447368,24.680702,0.989035,24.410080
53,203507,103,32.330097,29.407767,11.320388,18.611650,6.485437,10.330097,2.378641,8.786408,6.106796,0.893204,0.990291,3.116505,2.339806,25.479612,0.898058,22.882176
256,1630178,122,37.918033,27.450820,9.598361,21.245902,5.131148,5.786885,0.319672,3.483607,6.377049,1.811475,0.622951,2.418033,2.188525,20.842623,1.053279,21.953091
38,203081,58,36.103448,24.948276,7.655172,17.103448,6.241379,6.775862,0.500000,4.189655,7.068966,1.206897,0.172414,2.793103,1.672414,20.244828,1.002874,20.303002
4,201142,140,36.492857,26.235714,9.342857,17.857143,5.078571,5.914286,0.457143,5.257143,4.535714,0.800000,1.057143,3.114286,1.757143,19.933571,1.013690,20.206472
242,1630162,140,35.728571,28.100000,9.450000,20.307143,5.450000,6.664286,0.692857,4.678571,4.178571,1.242857,0.714286,3.021429,1.914286,19.948571,0.992460,19.798166
126,1628369,88,35.738636,25.909091,8.863636,19.909091,4.806818,5.897727,0.636364,8.261364,5.863636,1.113636,0.465909,2.818182,2.079545,19.900000,0.992740,19.755524
159,1628973,139,35.223022,26.021583,9.165468,19.244604,5.194245,6.251799,0.417266,2.705036,7.035971,0.841727,0.115108,2.438849,2.215827,19.419424,0.978417,19.000300


## 11. Rookie Valuation

**Question:** How do we assign value to incoming rookies who have zero historical NBA data, without unfairly punishing teams that rely heavily on recent lottery picks?

We use the NBA API's `HOW_ACQUIRED` feature to extract each player's original draft position. We then map their draft slot to a projected baseline value, rewarding top-3 picks with instant rotation impact while slotting lower picks closer to replacement level.


In [23]:
conditions = [
    roster_with_value_filtered['HOW_ACQUIRED'].str.contains(r'#(1|2|3)\b\s*Pick', case=False, na=False),
    roster_with_value_filtered['HOW_ACQUIRED'].str.contains(r'#([4-9]|1[0-4])\b\s*Pick', case=False, na=False),
    roster_with_value_filtered['HOW_ACQUIRED'].str.contains(r'#(1[5-9]|2[0-9]|30)\b\s*Pick', case=False, na=False)
]

# Top 3 pick = 9.0 (Solid Rotation)
# Lottery = 6.0 (Role Player)
# Late 1st = 4.0 (Bench Rotation)
values = [9.0, 6.0, 4.0]


rookie_proj = np.select(conditions, values, default=1.0)


# applying the proj value only to players who currently have a 0
rookie_mask = roster_with_value_filtered['PLAYER_VALUE'] == 0
roster_with_value_filtered.loc[rookie_mask, 'PLAYER_VALUE'] = rookie_proj[rookie_mask]

display(roster_with_value_filtered[rookie_mask][
    ['PLAYER', 'TeamID', 'HOW_ACQUIRED', 'PLAYER_VALUE']
].sort_values('PLAYER_VALUE', ascending=False).head(15))


C:\Users\Lian Paredes\AppData\Local\Temp\ipykernel_6960\3714969494.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  roster_with_value_filtered['HOW_ACQUIRED'].str.contains(r'#(1|2|3)\b\s*Pick', case=False, na=False),
C:\Users\Lian Paredes\AppData\Local\Temp\ipykernel_6960\3714969494.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  roster_with_value_filtered['HOW_ACQUIRED'].str.contains(r'#([4-9]|1[0-4])\b\s*Pick', case=False, na=False),
C:\Users\Lian Paredes\AppData\Local\Temp\ipykernel_6960\3714969494.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  roster_with_value_filtered['HOW_ACQUIRED'].str.contains(r'#(1[5-9]|2[0-9]|30)\b\s*Pick', case=False, na=False)


,PLAYER,TeamID,HOW_ACQUIRED,PLAYER_VALUE
490,Darryn Peterson,1610612762,#2 Pick in 2026 Draft,9.0
524,AJ Dybantsa,1610612764,#1 Pick in 2026 Draft,9.0
515,Cameron Boozer,1610612763,#3 Pick in 2026 Draft,9.0
90,Noa Essengue,1610612741,#12 Pick in 2025 Draft,6.0
7,Kingston Flemings,1610612737,#8 Pick in 2026 Draft,6.0
84,Caleb Wilson,1610612741,#4 Pick in 2026 Draft,6.0
575,Hannes Steinbach,1610612766,#14 Pick in 2026 Draft,6.0
407,Darius Acuff Jr.,1610612758,#7 Pick in 2026 Draft,6.0
279,Mikel Brown Jr.,1610612751,#6 Pick in 2026 Draft,6.0
448,Aday Mara,1610612760,#12 Pick in 2026 Draft,6.0


**Takeaway:** Teams with heavy incoming rookie talent (like San Antonio and Atlanta) no longer receive a mathematical `0` for those rotation spots. The roster now accurately reflects the baseline expected value of lottery picks entering the 2026-27 season.


In [24]:
# calculating value over replacement player (VORP)
REPLACEMENT_LEVEL = 3.0
roster_with_value_filtered['PLAYER_VALUE'] = (roster_with_value_filtered['PLAYER_VALUE'] - REPLACEMENT_LEVEL).clip(lower=0)

# player level verification
print("--- TOP 15 INDIVIDUAL PLAYERS (VERIFICATION) ---")
display(roster_with_value_filtered[['PLAYER', 'TeamID', 'RAW_GmSc', 'AVG_MIN', 'PLAYER_VALUE']]
        .sort_values('PLAYER_VALUE', ascending=False)
        .head(15))


# NOW  we rank the top 9 and sum the teams
roster_with_value_filtered['TEAM_RANK'] = roster_with_value_filtered.groupby('TeamID')['PLAYER_VALUE'].rank(ascending=False, method='first')
top_rotation_only = roster_with_value_filtered[roster_with_value_filtered['TEAM_RANK'] <= 9]


team_roster_value_v2 = top_rotation_only.groupby('TeamID').agg(
    TOTAL_ROSTER_VALUE=('PLAYER_VALUE', 'sum')
).reset_index()


team_roster_value_v2 = pd.merge(
    team_roster_value_v2, 
    team_id_to_abbrev, 
    on='TeamID', 
    how='left'
)

print("\n--- FINAL TEAM ROSTER VALUATION ---")
display(team_roster_value_v2.sort_values('TOTAL_ROSTER_VALUE', ascending=False))

team_roster_value_v2.to_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\roster_value_snapshot.csv', index=False)

--- TOP 15 INDIVIDUAL PLAYERS (VERIFICATION) ---


,PLAYER,TeamID,RAW_GmSc,AVG_MIN,PLAYER_VALUE
124,Nikola Jokić,1610612743,29.422963,35.851852,26.301881
439,Shai Gilgeous-Alexander,1610612760,26.770139,33.729167,22.081513
218,Luka Dončić,1610612747,24.680702,35.605263,21.410080
225,Giannis Antetokounmpo,1610612748,25.479612,32.330097,19.882176
348,Tyrese Maxey,1610612755,20.842623,37.918033,18.953091
387,Damian Lillard,1610612757,20.244828,36.103448,17.303002
162,Kevin Durant,1610612745,19.933571,36.492857,17.206472
265,Anthony Edwards,1610612750,19.948571,35.728571,16.798166
27,Jayson Tatum,1610612738,19.900000,35.738636,16.755524
306,Jalen Brunson,1610612752,19.419424,35.223022,16.000300



--- FINAL TEAM ROSTER VALUATION ---


,TeamID,TOTAL_ROSTER_VALUE,TEAM_ABBREVIATION
18,1610612755,82.044317,PHI
6,1610612743,68.571142,DEN
20,1610612757,67.944996,POR
15,1610612752,66.244100,NYK
8,1610612745,66.043078,HOU
3,1610612740,62.675562,NOP
16,1610612753,62.536753,ORL
27,1610612764,61.969383,WAS
22,1610612759,60.304392,SAS
23,1610612760,60.178900,OKC


## 11. Phase 2 Summary

This notebook produced three outputs:

1. **`games_regular_season_grouped`** — team-season level table covering 
   2024-25 and 2025-26, with standings, Four Factors, Net Rating, MVP 
   shares, All-NBA presence, and playoff depth/championship status. 
   This is the training feature table for Phase 3.

2. **`team_playoff_experience_final`** — a current-roster snapshot 
   (as of today (August 10, 2026)) of cumulative playoff experience per team, to be 
   refreshed in September and joined onto the 2026-27 prediction input.
3. **`roster_value_snapshot.csv`** — The current 2026-27 roster valuation for each team, aggregating post-VORP player Game Scores.
